In [ ]:
import re
import pandas as pd
from datetime import datetime

In [ ]:
class FakeLLMClient:
    def generate(self, prompt: str) -> str:
        return f"Ответ модели на обработанный запрос: {prompt}"

In [ ]:
class InputFilter:
    def __init__(self):
        self.dangerous_patterns = {
            "prompt_injection": [
                r"ignore previous instructions",
                r"ignore all instructions",
                r"system prompt",
                r"developer message",
                r"reveal your instructions",
                r"show hidden prompt",
                r"обойди ограничения",
                r"игнорируй предыдущие инструкции",
                r"покажи системный промпт",
                r"раскрой инструкции"
            ],
            "data_extraction": [
                r"training data",
                r"personal data",
                r"выведи данные пользователей",
                r"покажи персональные данные",
                r"раскрой конфиденциальную информацию"
            ],
            "jailbreak": [
                r"jailbreak",
                r"DAN",
                r"do anything now",
                r"без ограничений",
                r"режим разработчика"
            ]
        }

    def check(self, prompt: str) -> dict:
        lowered = prompt.lower()
        detected = []

        for attack_type, patterns in self.dangerous_patterns.items():
            for pattern in patterns:
                if re.search(pattern, lowered):
                    detected.append(attack_type)

        if detected:
            return {
                "allowed": False,
                "risk": "high",
                "detected_types": list(set(detected)),
                "reason": "Обнаружены признаки потенциально опасного запроса"
            }

        return {
            "allowed": True,
            "risk": "low",
            "detected_types": [],
            "reason": "Запрос не содержит явных признаков атаки"
        }